In [1]:
import pandas as pd
import duckdb

In [107]:
df = pd.read_csv("online_retail.csv" , parse_dates=["InvoiceDate"])
df.shape

(541909, 8)

In [5]:
conn = duckdb.connect()
conn.register("retail" , df)

In [14]:
result = conn.execute("SELECT * FROM retail LIMIT 1").fetchdf()
print(type(result))

<class 'pandas.core.frame.DataFrame'>


In [15]:
conn.execute("SELECT * FROM retail LIMIT 1").fetchdf()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom


In [21]:
result = conn.execute("""
SELECT 
    Country,
    SUM(Quantity) as total_quantity
FROM retail
GROUP BY Country
ORDER BY total_quantity DESC
""").fetchdf()

In [22]:
print(type(result))

<class 'pandas.core.frame.DataFrame'>


In [23]:
result.head()

,Country,total_quantity
0,United Kingdom,4263829.0
1,Netherlands,200128.0
2,EIRE,142637.0
3,Germany,117448.0
4,France,110480.0


In [24]:
result["total_quantity"].mean()
#using pandas
#That's the interoperability

np.float64(136222.36842105264)

In [25]:
# ALways ask 
# Which tool should perform this particular operation?
#E.g
    # Messy CSV
    #    ↓
    # DuckDB → heavy SQL analytics
    #    ↓
    # Pandas → Python-specific manipulation
    #    ↓
    # ML / visualization / application logic


## Deep interoperability

In [26]:
# .execute()
# gives a duckdb result/relation object 
conn.execute("DESCRIBE SELECT * FROM retail")

In [31]:
# .fetchall() -- gives a python list
result = conn.execute("""
SELECT 
    Country,
    SUM(Quantity) as total_quantity
FROM retail
GROUP BY Country
ORDER BY total_quantity DESC
LIMIT 5
""").fetchall()

print(type(result))
print(result)

<class 'list'>
[('United Kingdom', 4263829), ('Netherlands', 200128), ('EIRE', 142637), ('Germany', 117448), ('France', 110480)]


In [34]:
#.fetchdf() -- Pandas dataframe
result = conn.execute("""
SELECT 
    Country,
    SUM(Quantity) as total_quantity
FROM retail
GROUP BY Country
ORDER BY total_quantity DESC
LIMIT 4
""").fetchdf()

print(type(result))
print(result)

<class 'pandas.core.frame.DataFrame'>
          Country  total_quantity
0  United Kingdom       4263829.0
1     Netherlands        200128.0
2            EIRE        142637.0
3         Germany        117448.0


In [35]:
#then you can perform pandas operation on it
result.tail()

,Country,total_quantity
0,United Kingdom,4263829.0
1,Netherlands,200128.0
2,EIRE,142637.0
3,Germany,117448.0


### Opposite direction

In [43]:
data = {
    "Product" : ['Banana' , 'Apple' , 'Orange' , 'Coconut'],
    "Price" : [3000 , 1000 , 400 , 40],
    "Quantity" : [4 , 5, 9 , 9]
}

In [44]:
df = pd.DataFrame(data)

In [45]:
df.head()


,Product,Price,Quantity
0,Banana,3000,4
1,Apple,1000,5
2,Orange,400,9
3,Coconut,40,9


In [47]:
conn.register("Products" , df)
# .register() -- > "Make this Python object available to DuckDB under this name."

In [50]:
result = conn.execute("""
SELECT 
    Product,
    Price * Quantity AS Revenue
FROM Products    
""").df()

In [53]:
result["Revenue"].sum()

np.int64(20960)

# Day 3

In [54]:
# Business question
#       ↓
# Translate into SQL logic
#       ↓
# DuckDB
#       ↓
# Result
#       ↓
# Interpretation


In [57]:
#firstly understand the dataset
conn.execute("DESCRIBE SELECT * FROM retail").df()

,column_name,column_type,null,key,default,extra
0,InvoiceNo,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP_NS,YES,None,None,None
5,UnitPrice,DOUBLE,YES,None,None,None
6,CustomerID,DOUBLE,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None


In [60]:
conn.execute("SELECT * FROM retail LIMIT 1").df()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom


### Pure applications to answer business questions

In [64]:
#1. Revenue == Quantity * UNIt price
conn.execute("""
SELECT
    Country,
    SUM(Quantity * UnitPrice) AS Revenue
FROM retail
GROUP BY Country
ORDER BY Revenue DESC
LIMIT 5    
""").df()

,Country,Revenue
0,United Kingdom,8.187806e+06
1,Netherlands,2.846615e+05
2,EIRE,2.632768e+05
3,Germany,2.216982e+05
4,France,1.974039e+05


In [68]:
#top products 
conn.execute("""
    SELECT
        Description,
        SUM(Quantity * UnitPrice) AS revenue
    FROM retail
    GROUP BY Description
    ORDER BY revenue DESC
    LIMIT 10
""").fetchdf()

,Description,revenue
0,DOTCOM POSTAGE,206245.48
1,REGENCY CAKESTAND 3 TIER,164762.19
2,WHITE HANGING HEART T-LIGHT HOLDER,99668.47
3,PARTY BUNTING,98302.98
4,JUMBO BAG RED RETROSPOT,92356.03
5,RABBIT NIGHT LIGHT,66756.59
6,POSTAGE,66230.64
7,PAPER CHAIN KIT 50'S CHRISTMAS,63791.94
8,ASSORTED COLOUR BIRD ORNAMENT,58959.73
9,CHILLI LIGHTS,53768.06


In [74]:
#top products by quantity
conn.execute("""
SELECT 
    Description,
    SUM(Quantity) as Total_quant
FROM retail
GROUP BY Description
ORDER BY Total_quant DESC
LIMIT 5
""").df()

,Description,Total_quant
0,WORLD WAR 2 GLIDERS ASSTD DESIGNS,53847.0
1,JUMBO BAG RED RETROSPOT,47363.0
2,ASSORTED COLOUR BIRD ORNAMENT,36381.0
3,POPCORN HOLDER,36334.0
4,PACK OF 72 RETROSPOT CAKE CASES,36039.0


In [100]:

#Average of the total prices per product
result = conn.execute("""
SELECT 
    Description,
    SUM(UnitPrice) AS Total_price
FROM retail
GROUP BY Description
""").df()
result["Total_price"].mean()

np.float64(591.5729105113637)

In [95]:
# aVERAGE SELLING PRICE
conn.execute("""
SELECT
    AVG(UnitPrice) AS average_price
FROM retail
Limit 5
""").df()

,average_price
0,4.611114


In [82]:
conn.execute("""
    SELECT
        Country,
        AVG(UnitPrice) AS average_unit_price
    FROM retail
    GROUP BY Country
    ORDER BY average_unit_price DESC
    LIMIT 5
""").fetchdf()

,Country,average_unit_price
0,Singapore,109.645808
1,Hong Kong,42.505208
2,Portugal,8.582976
3,Cyprus,6.302363
4,Canada,6.030331


In [101]:
# No. of transactions
conn.execute("""
SELECT
    COUNT(DISTINCT InvoiceNo) AS number_of_invoices
FROM retail    
""").df()

,number_of_invoices
0,25900


In [102]:
#Monthly Revenue
conn.execute("""
SELECT 
    DATE_TRUNC('month' , InvoiceDate) AS month,
    SUM(UnitPrice * Quantity) AS Total_Revenue
FROM retail
GROUP BY month
ORDER BY Total_Revenue DESC
LIMIT 5
""").df()

,month,Total_Revenue
0,2011-11-01,1461756.250
1,2011-10-01,1070704.670
2,2011-09-01,1019687.622
3,2010-12-01,748957.020
4,2011-05-01,723333.510


In [103]:
# How many unique customers are in the dataset?
conn.execute("""
SELECT
    COUNT(DISTINCT CustomerID) AS number_of_invoices
FROM retail    
""").df()

,number_of_invoices
0,4372


In [105]:
# How much revenue did each customer generate?
conn.execute("""
SELECT 
    CustomerID,
    SUM(UnitPrice * Quantity) AS Customer_Revenue
FROM retail
GROUP BY CustomerID
ORDER BY Customer_Revenue DESC
LIMIT 5
""").df()

,CustomerID,Customer_Revenue
0,NaN,1447682.12
1,14646.0,279489.02
2,18102.0,256438.49
3,17450.0,187482.17
4,14911.0,132572.62
